# T-Transformer training (Colab GPU)

Trains the from-scratch transformer in `model.py` as a translator, one model per
language pair (ca-en, de-en), on `opus_books`. Use a **GPU runtime**
(Runtime -> Change runtime type -> T4 GPU). Artifacts land in `models/<pair>/`,
get versioned to Drive, and are zipped for download into the local repo.

In [ ]:
# Cell 1 - mount Drive and enter the project
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/Transformer")  # adjust if nested
if not ROOT.exists():
    raise FileNotFoundError(f"Not found: {ROOT}. Upload the repo to Drive first.")

os.chdir(ROOT)
print("CWD:", Path.cwd())
print("train.py exists:", (Path.cwd() / "train.py").exists())
print("model.py exists:", (Path.cwd() / "model.py").exists())

In [ ]:
# Cell 2 - install dependencies (training set)
!pip -q install -U torch tokenizers datasets evaluate sacrebleu

In [ ]:
# Cell 3 - preflight
import torch, sys
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, f"| VRAM {p.total_memory/1024**3:.1f} GB")
else:
    print("WARNING: no GPU. Switch to a T4 runtime for a real training run.")

In [ ]:
# Cell 4 - training config (env-driven; tuned for a T4)
import os

# QUICK_RUN=1 does a fast sanity pass; set to 0 for the full run.
os.environ["QUICK_RUN"] = os.environ.get("QUICK_RUN", "0")
os.environ["QUICK_SAMPLES"] = "2000"

os.environ["EPOCHS"] = "25"
os.environ["BATCH_SIZE"] = "32"
os.environ["LR"] = "7e-4"
os.environ["WARMUP_STEPS"] = "2000"
os.environ["D_MODEL"] = "256"
os.environ["N_LAYERS"] = "4"
os.environ["N_HEADS"] = "8"
os.environ["D_FF"] = "512"
os.environ["SEQ_LEN"] = "64"
os.environ["VOCAB_SIZE"] = "16000"
print({k: os.environ[k] for k in ("QUICK_RUN", "EPOCHS", "BATCH_SIZE", "LR", "D_MODEL")})

In [ ]:
# Cell 5 - train both pairs (writes models/ca_en and models/de_en)
!python train.py --pairs ca-en de-en

In [ ]:
# Cell 6 - version artifacts to Drive + build a download zip
from datetime import datetime, timezone
from pathlib import Path
import json, shutil

SERVABLE = ("weights.pth", "config.json", "src_tokenizer.json", "tgt_tokenizer.json")
models_dir = Path("models").resolve()
pairs = [p for p in models_dir.iterdir() if p.is_dir() and (p / "config.json").exists()]
if not pairs:
    raise FileNotFoundError("No trained pairs under models/. Run Cell 5 first.")

version_tag = datetime.now(timezone.utc).strftime("v%Y%m%d-%H%M%S")
summary = {"version": version_tag, "pairs": {}}

for pair_dir in pairs:
    snapshot = pair_dir / "versions" / version_tag
    snapshot.mkdir(parents=True, exist_ok=True)
    for name in SERVABLE:
        shutil.copy2(pair_dir / name, snapshot / name)
    (pair_dir / "LATEST").write_text(str(snapshot))
    summary["pairs"][pair_dir.name] = json.loads((pair_dir / "config.json").read_text()).get("metrics", {})

(models_dir / "metrics.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

# stage only servable files (exclude versions/) and zip
staging = Path("/content/models_export")
if staging.exists():
    shutil.rmtree(staging)
for pair_dir in pairs:
    dst = staging / pair_dir.name
    dst.mkdir(parents=True, exist_ok=True)
    for name in SERVABLE:
        shutil.copy2(pair_dir / name, dst / name)
zip_path = shutil.make_archive("/content/models_artifact", "zip", staging)
print("zip:", zip_path)

In [ ]:
# Cell 7 - sanity-check the trained artifact with the real serving code
import importlib, serve_model
importlib.reload(serve_model)
reg = serve_model.build_registry("models", serve_model.select_device())
print("supported:", reg.supported_langs)
if "ca" in reg.supported_langs:
    print("CA->EN:", reg.get("ca").translate("El perill era desesperat.", num_beams=3))
if "de" in reg.supported_langs:
    print("DE->EN:", reg.get("de").translate("Das ist ein Test.", num_beams=3))

In [ ]:
# Cell 8 - download the artifact zip to your local machine
from google.colab import files
files.download("/content/models_artifact.zip")
# Then locally:  unzip models_artifact.zip -d models/